<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Deep-Learning/03-neural-network-building-blocks.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **神经网络基本构件** {#neural-network-building-blocks}

现代神经网络可以包含数十亿个参数，但其中的大部分计算仍由一组有限的基本构件组合而成：仿射变换负责混合特征，非线性函数构造具有表达力的决策边界，嵌入层把离散身份转换成可学习向量，输出头把通用表示翻译成具体任务的预测，而残差路径、门控与归一化则调节信息在网络中的流动方式。

分别理解这些构件有三个重要价值。第一，它能把一张复杂的架构图还原成一组可以审查的张量契约。第二，它能解释为什么两个输入、输出形状相同的网络，训练行为却可能截然不同。第三，它使实现具备模块化：一个新模型往往不是一套完全陌生的算法，而是对已有模块更谨慎的重新排列与组合。

本章统一使用 $B$ 表示批量大小，$L$ 表示序列长度，$D$ 表示特征宽度，$V$ 表示词表或类别表大小，$K$ 表示输出类别数。面对每一个模块，都应回答以下问题：

1. 它接收什么信息，又返回什么信息？
2. 它混合、保留或归一化了哪些轴？
3. 哪些量是学习得到的参数，哪些量是由当前数据产生的激活值？
4. 当模块被遗漏、放错位置或收到不兼容的形状时，会出现什么故障？

### **人工神经元与线性变换** {#artificial-neurons-linear-transformations}

一个**人工神经元**先把输入特征组合成一个标量，再根据需要通过非线性激活函数。对于输入向量 $\mathbf{x} \in \mathbb{R}^{D_{in}}$，

$$
z = \mathbf{w}^{\top}\mathbf{x} + b,
\qquad
y = \phi(z).
$$

权重 $\mathbf{w}$ 决定各个特征贡献的方向和强度，偏置 $b$ 移动响应阈值，$\phi$ 则决定预激活值 $z$ 如何变成传给下一层的激活值 $y$。可以把它类比成一次加权表决：每个特征投出一个带符号的票，偏置代表预先存在的倾向，激活函数决定如何向后续层公开最终分数。

一个神经层通常会并行计算许多神经元。对于小批量 $X \in \mathbb{R}^{B \times D_{in}}$ 和 $D_{out}$ 个输出神经元，

$$
Z = XW^{\top} + \mathbf{b},
\qquad
W \in \mathbb{R}^{D_{out} \times D_{in}},
\qquad
\mathbf{b} \in \mathbb{R}^{D_{out}}.
$$

输出形状为 `[B,D_out]`。每个输出神经元都接收全部输入特征，因此这种运算被称为**全连接层**或**稠密层**。严格从数学上看，$XW^{\top}$ 是线性映射，而加入非零偏置后的完整映射是仿射映射。不过在深度学习的日常术语中，`Linear` 通常指整个仿射运算。

从几何角度看，标量预激活 $z=\mathbf{w}^{\top}\mathbf{x}+b$ 衡量输入位于超平面 $\mathbf{w}^{\top}\mathbf{x}+b=0$ 哪一侧以及距离多远。因此，单个神经元能够表示线性决策边界，却无法独立表示不连通区域、XOR 关系，或一个会随上下文改变影响方向的特征。这些限制正是需要多层组合和非线性激活的原因。

<details>
<summary><strong>PyTorch：根据存储参数复现 <code>nn.Linear</code></strong></summary>

~~~python
import torch
from torch import nn

torch.manual_seed(7)

B, D_IN, D_OUT = 3, 4, 2
features = torch.randn(B, D_IN)
linear = nn.Linear(D_IN, D_OUT, bias=True)

# PyTorch stores weight as [D_out, D_in], so the batch uses weight.T.
manual_output = features @ linear.weight.T + linear.bias
module_output = linear(features)

assert linear.weight.shape == (D_OUT, D_IN)
assert linear.bias.shape == (D_OUT,)
assert module_output.shape == (B, D_OUT)
assert torch.allclose(manual_output, module_output)

print("input:", tuple(features.shape))
print("weight:", tuple(linear.weight.shape))
print("output:", tuple(module_output.shape))
~~~

</details>

**应用。** 线性变换可以充当分类器、回归层、特征投影、注意力中的 query/key/value 投影、通道混合器，以及 Transformer 前馈网络中的扩张和收缩映射。它不一定直接产生最终预测；更常见的作用是旋转、组合或调整中间表示的宽度，使后续模块能够处理这些表示。

**对比总结。** 一个神经元产生一个加权响应，稠密层则同时计算多个神经元。无偏置映射保持原点并且严格线性，带偏置映射是仿射的。线性层能够沿特征轴混合信息，但如果没有非线性，继续堆叠线性层也不会得到真正的非线性模型。

### **多层感知机** {#multilayer-perceptrons}

**多层感知机（MLP）**交替组合仿射变换和非线性激活函数。对于单隐藏层网络，

$$
H = \phi(XW_1^{\top}+\mathbf{b}_1),
\qquad
O = HW_2^{\top}+\mathbf{b}_2.
$$

其中 $X \in \mathbb{R}^{B \times D_{in}}$、$H \in \mathbb{R}^{B \times D_h}$、$O \in \mathbb{R}^{B \times D_{out}}$。隐藏宽度 $D_h$ 控制这一层能够并行表示多少个中间特征。隐藏表示并不是由人手工指定的；训练过程会发现哪些特征组合能够让最终任务变得更容易。

![一个包含四个输入、五个隐藏单元和三个输出的全连接多层感知机。](assets/d2l-mlp.svg){fig-align="center" width="68%" fig-alt="一个由输入层、单个全连接隐藏层和输出层组成的多层感知机结构图。"}

*图片来源：[Dive into Deep Learning：Multilayer Perceptrons](https://d2l.ai/chapter_multilayer-perceptrons/mlp.html)，采用 [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) 许可。*

激活函数不可缺少。如果去掉它，两层仿射变换可以合并成一层：

$$
(XW_1^{\top}+\mathbf{b}_1)W_2^{\top}+\mathbf{b}_2
= X(W_2W_1)^{\top} + (\mathbf{b}_1W_2^{\top}+\mathbf{b}_2).
$$

此时，深度虽然改变了参数化方式，却没有改变模型能够表示的输入输出函数族。非线性的 $\phi$ 阻止了这种代数合并，使网络能够构造依赖上下文的特征。对于 ReLU 网络，每一种激活模式都对应一个局部仿射映射；大量不同模式会把输入空间划分成许多分段线性区域。

**宽度**增加一层中可用的并行特征数量，**深度**则以层次化方式组合特征，使后续单元能够依赖前面已经构造出的模式。通用逼近定理说明，一个足够宽的单隐藏层 MLP 能在有界区域内逼近很广泛的函数类别；但这只是存在性结论，并不保证优化过程容易、数据充足或参数效率合理。对于具有组合结构的函数，深层网络通常能用更紧凑的方式表示。

<details>
<summary><strong>PyTorch：追踪双隐藏层 MLP 中的形状变化</strong></summary>

~~~python
import torch
from torch import nn

torch.manual_seed(11)

mlp = nn.Sequential(
    nn.Linear(4, 8),   # [B, 4] -> [B, 8]
    nn.ReLU(),
    nn.Linear(8, 6),   # [B, 8] -> [B, 6]
    nn.GELU(),
    nn.Linear(6, 3),   # [B, 6] -> [B, 3] logits
)

shape_trace = []
handles = []


def record_shape(name):
    """Create a forward hook that records a module's output contract."""
    def hook(module, inputs, output):
        shape_trace.append((name, tuple(output.shape)))
    return hook


for index, layer in enumerate(mlp):
    handles.append(layer.register_forward_hook(record_shape(f"{index}:{layer.__class__.__name__}")))

batch = torch.randn(5, 4)
logits = mlp(batch)

for handle in handles:
    handle.remove()

assert logits.shape == (5, 3)
print(*shape_trace, sep="\n")
~~~

</details>

MLP 的表达能力很强，但它对输入结构几乎不作假设。如果把一张 $224 \times 224$ 的 RGB 图像展平，再全连接到 4,096 个隐藏单元，仅第一层就需要大约 $224 \cdot 224 \cdot 3 \cdot 4096 \approx 6.17$ 亿个权重。卷积、注意力和图算子会通过局部性、参数共享或关系结构来减少或重新组织这种成本。即便如此，MLP 仍然广泛存在于预测头和按通道计算的前馈模块中。

**应用。** MLP 是表格数据、低维信号、已学习特征向量和分类头的有力基线。在 Transformer 中，前馈子层会对每个 token 位置独立应用同一个小型 MLP；注意力先在位置之间混合信息，MLP 再在通道之间混合信息。

**对比总结。** 线性模型只构造一个仿射映射，MLP 则用非线性函数组合多个仿射映射。宽度提供并行特征，深度负责组合特征；当稠密的全对全混合效率过低时，特定架构的层会加入更有用的归纳偏置。

### **激活函数** {#activation-functions}

**激活函数**把预激活值转换成继续向前传递的信号。隐藏层激活通常必须是非线性的，否则整个深层堆叠仍然只是一个仿射映射。由于函数一般逐元素应用，张量形状通常不变：

$$
Z \in \mathbb{R}^{B \times D}
\xrightarrow{\phi}
H \in \mathbb{R}^{B \times D}.
$$

激活函数的选择会影响表达能力、梯度流、稀疏性、数值范围和计算成本。还需要区分隐藏层激活与输出链接函数：ReLU、GELU 和 SiLU 常用于网络内部，而 sigmoid 或 softmax 在输出端可能已经隐含在任务损失中。

![ReLU 函数保留正输入，并把负输入映射为零。](assets/d2l-relu.svg){fig-align="center" width="64%" fig-alt="修正线性单元函数图像：负输入部分为零，正输入部分为线性函数。"}

*图片来源：[Dive into Deep Learning：Activation Functions](https://d2l.ai/chapter_multilayer-perceptrons/mlp.html#activation-functions)，采用 [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/) 许可。*

| 激活函数 | 定义 | 值域 | 主要行为 | 常见用途或注意事项 |
|---|---|---|---|---|
| ReLU | $\max(0,x)$ | $[0,\infty)$ | 计算便宜、输出稀疏、正区间斜率为 1 | 单元如果长期处于负区间，可能不再活跃 |
| Leaky ReLU | $\max(x,\alpha x)$ | $(-\infty,\infty)$ | 在负区间保留一个较小斜率 | 增加一个斜率超参数 |
| Sigmoid | $\sigma(x)=1/(1+e^{-x})$ | $(0,1)$ | 可解释为门控或 Bernoulli 概率 | 在较大 $|x|$ 处饱和，而且不是零中心 |
| Tanh | $\tanh(x)$ | $(-1,1)$ | 零中心的有界信号 | 在较大 $|x|$ 处同样会饱和 |
| GELU | $x\Phi(x)$ | 近似无界 | 平滑、依赖输入大小的衰减 | 常用于 Transformer 的 MLP |
| SiLU / Swish | $x\sigma(x)$ | 近似无界 | 平滑且具有轻微非单调性 | 常用于现代 CNN 和门控 MLP |

ReLU 在负半轴的导数为 0，在正半轴为 1；它在零点的精确导数取决于约定，各框架会选择一个次梯度。Sigmoid 满足 $\sigma'(x)=\sigma(x)(1-\sigma(x))$，其最大值只有 $1/4$，因此多层饱和 sigmoid 会显著削弱梯度。GELU 与 SiLU 会保留较小的负响应并且变化平滑，但额外的计算并不意味着它们在所有模型和设备上都必然更好。

激活函数还会与初始化和归一化相互作用。针对 ReLU 类网络保持方差的初始化尺度，与面向线性或 tanh 激活的初始化尺度不同。因此，如果只更换激活函数而保持其他假设不变，前向激活统计和反向梯度统计都可能发生变化。

<details>
<summary><strong>PyTorch：比较激活值与局部导数</strong></summary>

~~~python
import torch
from torch.nn import functional as F

sample_points = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])
activation_functions = {
    "relu": F.relu,
    "leaky_relu": lambda x: F.leaky_relu(x, negative_slope=0.1),
    "sigmoid": torch.sigmoid,
    "tanh": torch.tanh,
    "gelu": F.gelu,
    "silu": F.silu,
}

for name, function in activation_functions.items():
    x = sample_points.clone().requires_grad_(True)
    y = function(x)

    # Summing asks autograd for dy_i / dx_i at every independent element.
    y.sum().backward()
    print(f"{name:11s} values={y.detach().round(decimals=3).tolist()}")
    print(f"{'':11s} slopes={x.grad.round(decimals=3).tolist()}")

# A negative ReLU input has no local gradient, while Leaky ReLU keeps a path.
negative = torch.tensor([-2.0], requires_grad=True)
F.relu(negative).backward()
relu_slope = negative.grad.item()

negative.grad.zero_()
F.leaky_relu(negative, negative_slope=0.1).backward()
leaky_slope = negative.grad.item()

assert relu_slope == 0.0
assert abs(leaky_slope - 0.1) < 1e-6
~~~

</details>

**应用。** 在许多卷积网络和简单 MLP 中，ReLU 仍是可靠的默认选择。GELU 常见于 BERT 类 Transformer，SiLU 和 SwiGLU 类机制则出现在大量现代架构中。Sigmoid 尤其适合门控或二分类输出链接，而不是所有隐藏层的通用默认选项。

**对比总结。** ReLU 简单并能产生稀疏输出；Leaky ReLU 在负区间保留梯度路径；sigmoid 与 tanh 限制信号范围但容易饱和；GELU 和 SiLU 提供平滑、依赖输入的衰减。激活函数应当与初始化、归一化、架构和部署约束一起选择，而不能只依据流行程度。

### **嵌入与学习表示** {#embeddings-learned-representations}

**嵌入层**把一个离散标识符映射成稠密的可学习向量。如果词表或类别表包含 $V$ 个项目，每个项目对应一个 $D$ 维表示，那么参数构成一个表：

$$
E \in \mathbb{R}^{V \times D}.
$$

对于项目索引 $i$，输出就是第 $i$ 行 $E_i$。该操作等价于用 one-hot 向量 $\mathbf{e}_i \in \mathbb{R}^{V}$ 乘以嵌入表：

$$
\mathbf{h}_i = \mathbf{e}_i^{\top}E,
$$

但查表操作不需要显式构造一个几乎全为零的向量，也不需要与所有行相乘。对于形状为 `[B, L]` 的 token ID，`nn.Embedding(V, D)` 返回 `[B, L, D]`。

这里的关键不仅是降维。标识符本身不存在有意义的算术几何关系：token 23 不会天然比 token 900 更接近 token 24。训练会建立一个连续表示空间，让与任务有关的项目形成相似方向或邻域。这个几何结构由目标函数决定，因此用于情感分析的嵌入，可能会以不同于句法任务或推荐任务的方式组织词语。

嵌入表也广泛用于语言以外的任务，例如用户 ID、商品 ID、表格类别特征、图节点，以及经线性投影后的图像 patch、时间位置或序列位置。**padding index** 可以保留一个不参与更新的行。对于罕见类别和未见类别，必须设计明确策略，例如 unknown token、哈希、子词组合或基于可观测特征的编码。

<details>
<summary><strong>PyTorch：验证查表等价性与 padding 行行为</strong></summary>

~~~python
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(13)

VOCAB_SIZE, EMBED_DIM = 7, 4
embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)
token_ids = torch.tensor([[1, 4, 0], [4, 2, 6]])  # [B=2, L=3]

lookup_vectors = embedding(token_ids)              # [2, 3, 4]
one_hot = F.one_hot(token_ids, num_classes=VOCAB_SIZE).float()
matrix_vectors = one_hot @ embedding.weight

assert lookup_vectors.shape == (2, 3, EMBED_DIM)
assert torch.allclose(lookup_vectors, matrix_vectors)
assert torch.equal(lookup_vectors[0, 1], lookup_vectors[1, 0])  # same ID, same row

# The padding row is excluded from gradient updates.
lookup_vectors.sum().backward()
assert torch.count_nonzero(embedding.weight.grad[0]) == 0
assert torch.count_nonzero(embedding.weight.grad[4]) > 0

print("token IDs:", tuple(token_ids.shape))
print("embedded sequence:", tuple(lookup_vectors.shape))
~~~

</details>

在语言模型中，输入嵌入矩阵有时会与输出词表投影进行**权重绑定**。系统不再学习两个彼此无关、形状近似为 `[V, D]` 的矩阵，而是让一个参数表同时承担输入查找和输出评分。这既减少参数，也把“读取 token”与“预测 token”使用的几何空间联系起来，但前提是维度和建模假设相互兼容。

**应用。** 嵌入允许模型随下游任务共同学习单词、子词、用户、商品、类别或位置的表示。当一个身份会重复出现，并能获得足够多有信息量的更新时，嵌入尤其有效。

**对比总结。** One-hot 编码完整保留身份信息，但维度很高且不存在学习得到的相似性；嵌入查找更紧凑且可训练。查表方法会记忆每个项目专有的向量，而由可观测特征构造的编码器更容易泛化到未见项目，实际系统经常同时使用二者。

### **输出头** {#output-heads}

网络主干产生表示，**输出头**把这个表示转换成具体任务所需的参数形式。将二者分离可以更清楚地理解迁移学习：一个主干可以同时支持多个输出头，新任务也可能只需要替换最后的头部。

假设主干返回 $H \in \mathbb{R}^{B \times D}$，常见契约如下：

| 任务 | 输出头结果 | 目标 | 常用训练损失 | 推理变换 |
|---|---|---|---|---|
| 多分类 | logits `[B, K]` | 类别索引 `[B]` | 交叉熵 | softmax 后取 argmax |
| 二分类 | logit `[B]` 或 `[B,1]` | 二值浮点目标 | 带 logits 的 BCE | sigmoid 后按阈值判断 |
| 多标签分类 | logits `[B, K]` | 二值矩阵 `[B,K]` | 逐元素带 logits 的 BCE | 独立 sigmoid 与阈值 |
| 回归 | 数值 `[B,R]` | 连续目标 `[B,R]` | MSE、MAE 或似然损失 | 通常为恒等变换 |
| Token 分类 | logits `[B,L,K]` | token 标签 `[B,L]` | 带掩码的 token 交叉熵 | 逐 token argmax 或结构化解码器 |
| 检索 | 嵌入 `[B,D_r]` | 配对或相关性标签 | 对比损失或排序损失 | 相似度搜索 |

**Logits** 是不受约束的分数。在 `CrossEntropyLoss` 前先应用 softmax，或在 `BCEWithLogitsLoss` 前先应用 sigmoid，是常见错误，因为这些损失已经把链接函数与数值稳定的对数损失计算组合在一起。除非另一个损失明确要求输入概率，否则概率转换应放在推理或解释阶段。

输出头同时编码了任务假设。$K$ 类 softmax 假设结果互斥且只能选择一个，而 $K$ 个 sigmoid 输出允许多个标签同时成立。标量回归头假设点估计足以描述响应；概率输出头则可以预测均值和尺度、分位数或分布参数，以表达不确定性。

<details>
<summary><strong>PyTorch：为同一个表示连接分类头与回归头</strong></summary>

~~~python
import torch
from torch import nn
from torch.nn import functional as F


class MultiTaskHeads(nn.Module):
    """Map one [B, D] representation to two task-specific outputs."""

    def __init__(self, feature_dim: int, number_of_classes: int):
        super().__init__()
        self.classifier = nn.Linear(feature_dim, number_of_classes)
        self.regressor = nn.Linear(feature_dim, 1)

    def forward(self, representation: torch.Tensor):
        return {
            "class_logits": self.classifier(representation),  # [B, K]
            "value": self.regressor(representation).squeeze(-1),  # [B]
        }


torch.manual_seed(17)
B, D, K = 6, 12, 4
representation = torch.randn(B, D)
heads = MultiTaskHeads(D, K)
predictions = heads(representation)

class_targets = torch.tensor([0, 3, 1, 2, 1, 0])
value_targets = torch.randn(B)

# Pass raw logits to cross-entropy; do not apply softmax first.
classification_loss = F.cross_entropy(predictions["class_logits"], class_targets)
regression_loss = F.mse_loss(predictions["value"], value_targets)
total_loss = classification_loss + 0.25 * regression_loss

assert predictions["class_logits"].shape == (B, K)
assert predictions["value"].shape == (B,)
assert total_loss.ndim == 0
~~~

</details>

当多个输出头共享特征时，各个损失的相对权重会决定哪一个任务更主导表示学习。相同的数值权重并不等于相同的梯度影响，因为不同损失的尺度、噪声和曲率可能不同。因此，多任务系统需要分别监控每个任务，并可能调整、归一化或动态适配损失权重。

**应用。** 一个视觉主干可以同时连接类别、边界框和分割输出头；语言编码器可以连接意图分类与 token 标注输出头；推荐模型可以同时预测点击概率和预期价值。输出头应精确暴露损失函数和评估流程真正消费的量。

**对比总结。** 主干学习可复用特征，输出头施加任务契约。Softmax 表示类别之间的竞争，sigmoid 独立处理各个标签，回归头预测连续量，嵌入头则优化表示空间的几何结构，而不是直接输出标签。

### **残差连接与跳跃连接** {#residual-skip-connections}

**跳跃连接**创建一条绕过一个或多个变换的路径。最常见的残差块计算：

$$
Y = X + F(X;\theta),
$$

其中 $F$ 是可学习的残差分支，快捷路径是恒等映射。分支不必从头复现完整目标映射 $H(X)$，而是参数化变化量 $F(X)=H(X)-X$。如果保留当前表示最合适，残差分支可以趋近于零，而恒等路径仍然保持可用。

![原始 ResNet 残差块把恒等快捷路径与两层可学习残差分支相加。](assets/resnet-residual-block.png){fig-align="center" width="58%" fig-alt="原始 ResNet 残差基本块：一条分支经过两个权重层，另一条恒等快捷路径在 ReLU 前与其相加。"}

*图片来源：He 等人，[Deep Residual Learning for Image Recognition](https://openaccess.thecvf.com/content_cvpr_2016/html/He_Deep_Residual_Learning_CVPR_2016_paper.html)，图 2。*

加法要求两个分支的形状相互兼容。如果 $F(X)$ 改变了宽度、通道数或空间分辨率，快捷路径也必须变换：

$$
Y = P(X) + F(X),
$$

其中 $P$ 可以是可学习的线性投影、带步幅卷积，或确定性的填充与下采样操作。恒等快捷路径不增加参数，投影快捷路径则会增加参数。

残差路径有助于优化，因为它为激活信号和敏感度信号都提供了一条直接通道。直观地说，对 $Y=X+F(X)$ 关于 $X$ 求导时，除了经过 $F$ 的导数之外，还包含一个恒等项。第 04 章会对此进行严格推导。不过，残差连接本身并不能保证训练稳定：不合适的缩放、归一化位置错误或过大的残差更新，仍然可能使深层堆叠失稳。

归一化和激活函数的位置同样重要。在**后激活**块中，变换、相加和激活遵循原始 ResNet 的顺序。许多现代**预归一化**块则先在残差分支入口归一化输入，并让加法保持为干净的恒等更新。即使张量形状完全相同，这种顺序变化也会改变信号传播行为。

<details>
<summary><strong>PyTorch：实现带可选投影的 MLP 残差块</strong></summary>

~~~python
import torch
from torch import nn


class ResidualMLPBlock(nn.Module):
    """A pre-normalized residual block for [B, D] feature tensors."""

    def __init__(self, input_dim: int, output_dim: int, hidden_dim: int):
        super().__init__()
        self.norm = nn.LayerNorm(input_dim)
        self.branch = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )
        self.shortcut = (
            nn.Identity()
            if input_dim == output_dim
            else nn.Linear(input_dim, output_dim, bias=False)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual_update = self.branch(self.norm(x))
        return self.shortcut(x) + residual_update


same_width = ResidualMLPBlock(input_dim=16, output_dim=16, hidden_dim=32)
changed_width = ResidualMLPBlock(input_dim=16, output_dim=24, hidden_dim=32)
features = torch.randn(8, 16)

assert same_width(features).shape == (8, 16)
assert changed_width(features).shape == (8, 24)
assert sum(p.numel() for p in same_width.shortcut.parameters()) == 0
assert sum(p.numel() for p in changed_width.shortcut.parameters()) == 16 * 24
~~~

</details>

并非所有跳跃连接都使用加法。U-Net 会把编码器特征图传给解码器并进行拼接，DenseNet 会拼接多个较早表示，编码器-解码器模型也会跨越瓶颈传递多尺度信息。拼接会完整保留两路输入，却增加特征宽度和后续计算成本；加法保持宽度不变，但要求两个分支处于同一个坐标空间。

**应用。** 残差更新已经成为 ResNet、Transformer、扩散网络、状态空间模块和大型 MLP 架构的标准构件。当瓶颈可能丢失精细空间细节时，跨尺度跳跃连接尤其重要。

**对比总结。** 普通堆叠会在每个模块替换当前表示；残差加法学习一个同形状更新；投影残差负责协调维度；拼接式跳跃连接保留独立特征集合，但会产生更大的张量。

### **门控机制** {#gating-mechanisms}

**门控**是对信息流进行学习和数据依赖控制的机制。一个通用插值门可以写成：

$$
G = \sigma(A(X)),
\qquad
Y = G \odot U(X) + (1-G) \odot V(X),
$$

其中 $G$ 的值位于 $(0,1)$，$\odot$ 表示逐元素乘法。门控可以类比成连续可调的阀门：它可以保留一路信号、偏向另一路信号，或针对不同样本、token、通道和空间位置采用不同混合比例。

门控比固定残差相加更有表达力，因为路由强度取决于输入。但它也会引入额外投影和新的故障模式：sigmoid 门可能饱和到接近 0 或 1，使导数很小，导致已经形成的路由决定难以改变。有些架构会专门初始化门控偏置，让网络初始阶段倾向于保留信息，而不是压制信息。

若干常用模块都符合这一模式：

| 机制 | 公式 | 含义 |
|---|---|---|
| GLU | $\operatorname{GLU}(X)=A(X)\odot\sigma(B(X))$ | sigmoid 分支控制 value 分支 |
| SwiGLU | $\operatorname{SwiGLU}(X)=A(X)\odot\operatorname{SiLU}(B(X))$ | 平滑的门控前馈变换 |
| Highway 更新 | $T(X)\odot H(X)+(1-T(X))\odot X$ | 在变换信息与直接传递信息之间选择 |
| LSTM/GRU 门 | 多个 sigmoid 控制的状态更新 | 决定写入、擦除或暴露哪些序列状态 |
| 混合专家路由器 | 对专家分支的归一化分数 | 选择稀疏或稠密计算路径 |

“门”这个词并不意味着它在训练中必须严格取二值。普通可微训练通常使用连续控制值。硬路由或稀疏路由需要额外的梯度估计器、正则项或分发逻辑。

<details>
<summary><strong>PyTorch：实现 SwiGLU 前馈模块</strong></summary>

~~~python
import torch
from torch import nn
from torch.nn import functional as F


class SwiGLU(nn.Module):
    """A gated feed-forward block that preserves the final feature width."""

    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int):
        super().__init__()
        # One projection produces both the value and gate pre-activations.
        self.value_and_gate = nn.Linear(input_dim, 2 * hidden_dim)
        self.output = nn.Linear(hidden_dim, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        value, gate = self.value_and_gate(x).chunk(2, dim=-1)
        hidden = value * F.silu(gate)
        return self.output(hidden)


torch.manual_seed(19)
B, L, D = 2, 5, 16
block = SwiGLU(input_dim=D, hidden_dim=32, output_dim=D)
sequence = torch.randn(B, L, D)
output = block(sequence)

assert output.shape == (B, L, D)
assert block.value_and_gate.weight.shape == (64, D)
print("parameters:", sum(parameter.numel() for parameter in block.parameters()))
~~~

</details>

**应用。** 门控用于管理循环网络记忆、调节 highway network 的特征流、改进 Transformer 前馈层，以及在混合专家系统中路由 token。当模型需要决定“通过多少信息”或“走哪一条路径”，而不是始终应用相同更新时，门控最有价值。

**对比总结。** 激活函数变换信号，残差连接加入固定的结构路径，门控则学习依赖数据的乘子或混合比例。门控提供自适应控制，但会增加参数，而且如果初始化和监控不合适，可能发生饱和或路由坍缩。

### **归一化层** {#normalization-layers}

**归一化层**会标准化选定的一组激活值，并通常在之后应用可学习的缩放与平移。对于由归一化规则选出的值集合 $S$，

$$
\mu_S=\frac{1}{|S|}\sum_{i\in S}x_i,
\qquad
\sigma_S^2=\frac{1}{|S|}\sum_{i\in S}(x_i-\mu_S)^2,
$$

$$
\widehat{x}_i=\frac{x_i-\mu_S}{\sqrt{\sigma_S^2+\epsilon}},
\qquad
y_i=\gamma_i\widehat{x}_i+\beta_i.
$$

各种归一化方法的核心区别是如何定义 $S$：哪些样本、通道、位置或空间点共享同一个均值和方差。较小的 $\epsilon$ 防止除零，可学习的 $\gamma$ 与 $\beta$ 则让网络在标准化后重新恢复或调整有用的尺度。

![Batch、Layer、Instance 与 Group Normalization 会在特征图张量中聚合不同的元素集合。](assets/normalization-axes-comparison.png){fig-align="center" width="88%" fig-alt="Batch Normalization、Layer Normalization、Instance Normalization 与 Group Normalization 对比图，蓝色单元表示共同参与同一组归一化统计的值。"}

*图片来源：Wu 与 He，[Group Normalization](https://openaccess.thecvf.com/content_ECCV_2018/html/Yuxin_Wu_Group_Normalization_ECCV_2018_paper.html)，图 2。*

对于图像张量 `[B, C, H, W]`：

- **BatchNorm** 对每个通道沿 `[B, H, W]` 计算一组统计量。推理时通常使用训练期间累计的运行估计，因此训练/评估模式非常重要。
- **LayerNorm** 在每个样本内部沿选定特征轴计算统计量。在形状为 `[B, L, D]` 的序列模型中，它通常对每个 token 独立归一化最后的宽度 `D`。
- **InstanceNorm** 对每个样本、每个通道沿 `[H, W]` 计算统计量，常用于需要去除实例特定对比度的场景。
- **GroupNorm** 把通道划分成若干组，并在每个样本内部归一化各组。它不依赖 batch 统计量。
- **RMSNorm** 只按均方根进行缩放，不减去均值；它减少部分计算，同时保留相近的稳定作用。

归一化层并不等同于输入预处理。数据集归一化使用固定统计量，让原始特征具有可比较尺度；内部归一化层作用于持续变化的学习激活，参与模型计算图，并可能包含可训练参数或运行缓冲区。

<details>
<summary><strong>PyTorch：验证 BN、LN、IN 与 GN 各自归一化的轴</strong></summary>

~~~python
import torch
from torch import nn

torch.manual_seed(23)
B, C, H, W = 4, 6, 3, 3
x = torch.randn(B, C, H, W) * 3.0 + 5.0

# Disable affine parameters so the post-normalization means are easy to inspect.
batch_norm = nn.BatchNorm2d(C, affine=False, track_running_stats=False)
layer_norm = nn.LayerNorm((C, H, W), elementwise_affine=False)
instance_norm = nn.InstanceNorm2d(C, affine=False, track_running_stats=False)
group_norm = nn.GroupNorm(num_groups=3, num_channels=C, affine=False)

bn_output = batch_norm(x)
ln_output = layer_norm(x)
in_output = instance_norm(x)
gn_output = group_norm(x)

# Each assertion averages over exactly the axes used to estimate the mean.
assert torch.allclose(bn_output.mean(dim=(0, 2, 3)), torch.zeros(C), atol=1e-5)
assert torch.allclose(ln_output.mean(dim=(1, 2, 3)), torch.zeros(B), atol=1e-5)
assert torch.allclose(in_output.mean(dim=(2, 3)), torch.zeros(B, C), atol=1e-5)

grouped = gn_output.reshape(B, 3, C // 3, H, W)
assert torch.allclose(grouped.mean(dim=(2, 3, 4)), torch.zeros(B, 3), atol=1e-5)

print("BN means per channel:", bn_output.mean(dim=(0, 2, 3)).round(decimals=6))
print("GN means per sample/group:", grouped.mean(dim=(2, 3, 4)).round(decimals=6))
~~~

</details>

当 batch 具有代表性且足够大时，BatchNorm 的效果非常好；但较小或样本并不独立的 batch 会产生噪声统计，而且训练统计与推理统计可能出现偏差。LayerNorm 与 GroupNorm 不依赖跨样本统计，因此适合可变长度序列、小 batch 视觉任务，以及同步 batch 统计成本过高的分布式场景。

**应用。** BatchNorm 与卷积网络联系紧密；LayerNorm 和 RMSNorm 主导 Transformer 类架构；GroupNorm 适合检测、分割、扩散模型等小 batch 视觉工作负载；InstanceNorm 常用于与风格有关的图像生成。

**对比总结。** 所有归一化层都会重新缩放激活，但它们耦合的轴和维护的状态不同。BatchNorm 依赖 batch，并且训练与评估行为不同；LayerNorm、GroupNorm 和 InstanceNorm 使用逐样本统计；RMSNorm 则省略均值中心化。

### **参数计数与计算成本** {#parameter-counting-computational-cost}

参数量衡量可学习状态的存储规模，计算成本衡量给定输入形状下执行的工作量。二者有关，但不能相互替代。嵌入表可能包含大量参数，却只在每个样本中访问少数几行；一个没有参数的激活函数，仍然需要处理巨大激活张量中的每个元素。

对于从 $D_{in}$ 映射到 $D_{out}$ 的稠密层，启用偏置时：

$$
N_{params}=D_{out}D_{in}+D_{out}.
$$

处理 $N$ 个输入向量大约需要：

$$
N \cdot D_{in}D_{out}
$$

次乘加运算（MAC），再加上较低阶的偏置和激活计算。有些报告把一次 MAC 计作一个操作，另一些会把乘法和加法分别计作两个 FLOPs，因此资源报告必须说明计数约定。

常用的一阶估算公式包括：

| 模块 | 可训练参数 | 每个向量的主要计算量 |
|---|---:|---:|
| Linear $D_{in}\to D_{out}$ | $D_{in}D_{out}+D_{out}$ | $D_{in}D_{out}$ MACs |
| Embedding $V\times D$ | $VD$ | 每个 ID 查找 $D$ 个值 |
| 沿 $D$ 的 LayerNorm | 通常为 $2D$ | $O(D)$ 的归约和逐元素运算 |
| 两层 MLP $D\to H\to D$ | 约 $2DH$ | 约 $2DH$ MACs |
| SwiGLU $D\to 2H$，再 $H\to D$ | 约 $3DH$ | 约 $3DH$ MACs |

批量大小和序列长度会成倍增加计算量与激活内存，但不改变模型参数量。训练内存还包括梯度、优化器状态、保存的激活和临时工作区。如果长序列激活主导内存，占用参数的小幅下降可能几乎没有效果；相反，一个巨大的嵌入表可能主导 checkpoint 大小，却并不主导 FLOPs。

<details>
<summary><strong>PyTorch：统计参数，并使用 hook 估算 Linear 层 MACs</strong></summary>

~~~python
import torch
from torch import nn


def count_trainable_parameters(module: nn.Module) -> int:
    return sum(parameter.numel() for parameter in module.parameters() if parameter.requires_grad)


def estimate_linear_macs(module: nn.Module, sample: torch.Tensor) -> int:
    """Estimate only dense Linear MACs for one forward pass."""
    total_macs = 0
    handles = []

    def linear_hook(layer, inputs, output):
        nonlocal total_macs
        input_tensor = inputs[0]
        number_of_vectors = input_tensor.numel() // layer.in_features
        total_macs += number_of_vectors * layer.in_features * layer.out_features

    for child in module.modules():
        if isinstance(child, nn.Linear):
            handles.append(child.register_forward_hook(linear_hook))

    with torch.no_grad():
        module(sample)

    for handle in handles:
        handle.remove()
    return total_macs


network = nn.Sequential(
    nn.Linear(64, 128),
    nn.GELU(),
    nn.Linear(128, 10),
)
batch = torch.randn(32, 64)

parameter_count = count_trainable_parameters(network)
linear_macs = estimate_linear_macs(network, batch)
parameter_mebibytes_fp32 = parameter_count * 4 / 2**20

assert parameter_count == (64 * 128 + 128) + (128 * 10 + 10)
assert linear_macs == 32 * (64 * 128 + 128 * 10)

print("trainable parameters:", parameter_count)
print("Linear MACs for the batch:", linear_macs)
print("parameter payload in fp32 MiB:", round(parameter_mebibytes_fp32, 4))
~~~

</details>

基于 hook 的估算适合教学，但不是完整性能分析器。它没有计入激活、归一化、内存流量、算子融合、并行效率和硬件特定行为。实际延迟应当在目标设备上预热后，使用有代表性的形状进行测量；峰值内存也必须在真实训练或推理模式下观察。

**应用。** 参数公式能在训练前发现意外过大的输出头或嵌入表；MAC 与激活估算帮助选择宽度、深度、批量大小和序列长度；性能分析再判断理论瓶颈是否真正出现在目标硬件上。

**对比总结。** 参数量描述可学习状态，MACs/FLOPs 描述算术量，激活大小与内存流量影响运行时间和内存，而墙钟延迟反映完整软硬件系统。所有指标都应同时给出输入形状和计数约定。

### **使用可复用 PyTorch 模块构建网络** {#building-network-reusable-pytorch-modules}

一个可复用神经网络模块应当负责一个内聚的变换，并让张量契约清晰可见。良好的模块边界通常具备：

- 明确的输入与输出宽度；
- 已注册的参数、缓冲区和子模块；
- 只描述数据流而不包含训练循环策略的 `forward` 方法；
- 可以安全堆叠的形状保持模块；
- 可针对不同任务替换的输出头；
- 在 checkpoint、性能分析和报错轨迹中仍有含义的名称。

`nn.Sequential` 适合单一直线式流水线。当前向传播需要显式循环时，`nn.ModuleList` 可以注册可变长度模块集合。`nn.ModuleDict` 则注册带名称的备选分支或输出头。普通 Python list 或 dictionary 不会自动注册其中的模块，因此其参数可能不会出现在 `.parameters()`、设备迁移和状态字典中。

下面的小型架构组合了本章多个构件。Stem 把原始特征投影到共享宽度；每个残差块使用预归一化和 SwiGLU 残差更新；最后通过带名称的输出头，把最终表示转换成分类和回归输出。

<details>
<summary><strong>PyTorch：组合可复用残差主干与多个输出头</strong></summary>

~~~python
import torch
from torch import nn
from torch.nn import functional as F


class GatedResidualBlock(nn.Module):
    """Pre-norm SwiGLU residual block with contract [B, D] -> [B, D]."""

    def __init__(self, width: int, hidden_width: int, dropout: float = 0.0):
        super().__init__()
        self.norm = nn.LayerNorm(width)
        self.expand = nn.Linear(width, 2 * hidden_width)
        self.contract = nn.Linear(hidden_width, width)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        normalized = self.norm(x)
        value, gate = self.expand(normalized).chunk(2, dim=-1)
        update = self.contract(value * F.silu(gate))
        return x + self.dropout(update)


class MultiTaskNetwork(nn.Module):
    """Shared feature backbone with named task-specific output heads."""

    def __init__(
        self,
        input_dim: int,
        width: int,
        hidden_width: int,
        depth: int,
        number_of_classes: int,
    ):
        super().__init__()
        self.stem = nn.Linear(input_dim, width)
        self.blocks = nn.ModuleList(
            [GatedResidualBlock(width, hidden_width, dropout=0.1) for _ in range(depth)]
        )
        self.final_norm = nn.LayerNorm(width)
        self.heads = nn.ModuleDict(
            {
                "class_logits": nn.Linear(width, number_of_classes),
                "value": nn.Linear(width, 1),
            }
        )

    def forward(self, features: torch.Tensor):
        hidden = self.stem(features)                  # [B, D_in] -> [B, D]
        for block in self.blocks:
            hidden = block(hidden)                    # width is preserved
        representation = self.final_norm(hidden)
        return {
            "representation": representation,
            "class_logits": self.heads["class_logits"](representation),
            "value": self.heads["value"](representation).squeeze(-1),
        }


torch.manual_seed(29)
model = MultiTaskNetwork(
    input_dim=20,
    width=32,
    hidden_width=64,
    depth=3,
    number_of_classes=5,
)
batch = torch.randn(7, 20)
outputs = model(batch)

assert outputs["representation"].shape == (7, 32)
assert outputs["class_logits"].shape == (7, 5)
assert outputs["value"].shape == (7,)

# Registered submodules appear automatically in parameters and checkpoints.
state = model.state_dict()
assert "blocks.0.expand.weight" in state
assert "heads.class_logits.weight" in state
print("trainable parameters:", sum(p.numel() for p in model.parameters()))
~~~

</details>

这个架构有意保持任务无关。它不包含优化器、损失、批数据加载器或分类阈值，因为这些属于训练与评估系统。`forward` 返回原始 logits 和显式表示，使下游行为可以在没有隐藏后处理的情况下进行测试。

有价值的模块测试包括：在多个 batch 大小下检查形状、检查评估模式下的确定性、验证预期输入范围内输出有限、完成状态字典往返加载，以及通过一个简单损失确认所有预期参数都能收到梯度。第 04 章会通过自动微分进一步分析最后一项检查。

**应用。** 可复用模块支持消融实验、迁移学习、多个输出头、架构搜索、性能分析、checkpoint 迁移和团队协作。只要输入输出契约保持兼容，就可以替换其中一个模块而不改动整个系统。

**对比总结。** `Sequential` 表达直线式链条，`ModuleList` 注册由自定义控制流使用的模块，`ModuleDict` 注册带名称的分支。良好的组合会把架构与训练策略分离，并让形状、状态和任务输出头保持可检查。

### **章节对比与总结** {#chapter-comparison-summary}

如果把所有架构拆分成三种角色，神经网络会更容易理解：**表示变换**、**信息流控制**和**任务解释**。

| 基本构件 | 主要作用 | 是否保持形状 | 可学习状态 | 重点排查的故障 |
|---|---|---:|---:|---|
| 线性层 | 混合特征并调整宽度 | 仅当输入输出宽度相同时 | 权重与可选偏置 | 轴使用错误或参数量过度增长 |
| MLP | 组合非线性特征变换 | 可配置 | 多个稠密层 | 缺少激活或宽度/深度不合适 |
| 激活函数 | 增加非线性或有界控制 | 通常是 | 通常没有 | 饱和、失活单元、初始化不匹配 |
| 嵌入 | 把身份映射到学习几何空间 | 增加特征轴 | 查找表 | 未见项目、padding、词表过大 |
| 输出头 | 施加任务特定预测契约 | 依赖任务 | 投影参数 | logits、目标和损失配对错误 |
| 残差连接 | 保留并更新信息 | 是，除非使用投影 | 无参数或投影参数 | 形状不兼容或更新尺度不合适 |
| 门控 | 路由或调节信息 | 通常是 | 门控/value 投影 | 饱和或路由坍缩 |
| 归一化 | 标准化选定激活集合 | 是 | 缩放/平移；有时含运行状态 | 归一化轴错误或训练/评估不匹配 |

本章的主要结论如下：

1. 稠密神经元计算仿射特征组合；非线性激活才是防止深层堆叠合并成一个仿射映射的关键。
2. MLP 宽度控制并行特征容量，深度则把特征组合成结构逐渐增强的表示。
3. 激活函数同时改变前向信号统计和局部导数行为，因此必须与初始化和架构相匹配。
4. 嵌入把离散 ID 转换成可训练几何空间，而且查表比显式 one-hot 乘法更高效。
5. 输出头应产生损失函数明确要求的原始量；logits 与概率不能互换。
6. 残差路径提供显式的信息保留通道，门控则让路由强度依赖数据。
7. 各种归一化方法最根本的区别，是哪些轴共享统计量，以及行为是否依赖训练 batch。
8. 参数量、算术量、激活内存和实测延迟分别回答不同的资源问题。
9. 可复用 PyTorch 模块应当让形状契约和已注册状态清晰可见，同时把训练策略放在 `forward` 之外。

下一章将解释这些构件如何学习：局部导数如何沿计算图组合，反向模式自动微分如何传播信用，以及梯度检查如何发现断裂或数值不稳定的路径。